# Execution and liquidity provision

## Summary

Which apparent gains remain after liquidity, fees, latency and fully funded inventory accounting?

The default workspace is **synthetic**. Its outputs establish software behavior, not empirical market findings. Run `pml demo` first, then execute this notebook top to bottom. Observed data can replace the workspace without changing the calculation functions.

## Context and methods

Compare taker, static and inventory-skew strategies at 0/250/1,000 ms extra latency. The documented market delay is additional for takers. Passive fills join behind displayed depth, require verified trade evidence, and receive no cancellation credit. Queue alternatives are scenarios, not bounds.

### Key assumptions

No observed edge is claimed. Unusable books, unknown costs and unavailable labels must stay visible. Results below are derived or simulated; inspect the input provenance before interpreting them.

## Data

The immutable run manifest identifies the exact captured inputs. Set `PML_WORKSPACE` to a workspace with the same layout to analyze another capture.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
import plotly.express as px

from prediction_market_lab.research import execution_study, load_table
from prediction_market_lab.storage import verify_run

workspace = Path(os.environ.get("PML_WORKSPACE", "data/demo")).resolve()
descriptor = json.loads((workspace / "workspace.json").read_text())
run = workspace / descriptor["run"]
manifest = verify_run(run)
features = load_table(run, "features")
provenance = sorted(features.input_provenance.unique())
print(
	{
		"input_provenance": provenance,
		"input_sha256": manifest["input_sha256"],
		"feature_rows": len(features),
	}
)
px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = ["#315c8c", "#b9812b", "#78619e"]

## Results

These calculations call the same tested functions as the terminal and CLI reports.

In [ ]:
runs = {p.parent.name: p.parent for p in (workspace / "paper").glob("*/manifest.json")}
study = execution_study(runs)
scenarios = pd.DataFrame(study["scenarios"])
display(scenarios[["scenario", "fills", "fees", "net_pnl", "exposure"]])
for row in study["scenarios"]:
	if row["net_pnl"] is not None:
		from decimal import Decimal

		decomposed = (
			Decimal(row["realized_price_pnl"])
			+ Decimal(row["settlement_pnl"])
			+ Decimal(row["inventory_price_revaluation"])
			- Decimal(row["fees"])
		)
		assert abs(decomposed - Decimal(row["net_pnl"])) < Decimal("0.00000001")

In [ ]:
scenarios["net_pnl_numeric"] = pd.to_numeric(scenarios.net_pnl)
fig = px.bar(
	scenarios,
	x="scenario",
	y="net_pnl_numeric",
	color="strategy",
	title="Paper outcome by latency and queue scenario<br><sup>Synthetic inputs are not economic evidence</sup>",
	labels={
		"scenario": "Declared execution scenario",
		"net_pnl_numeric": "Simulated net P&L (virtual dollars)",
	},
)
fig.add_hline(y=0, line_color="#777777")
fig.show()

## Validation and limitations

The following tables and checks are part of the result, including insufficient samples and negative findings.

In [ ]:
print("\n".join(study["limitations"]))
display(load_table(run, "ledger").tail(12))

## Takeaways

Use this experiment to inspect measurement, timing and execution assumptions. For synthetic inputs, differences across models or strategies have no external validity. For observed inputs, report the covered events, exclusions, uncertainty and sensitivity before making a market claim.

Method and sources: [research protocol](../docs/research-protocol.md), [data dictionary](../docs/data-dictionary.md), [source register](../docs/sources.md).